In [21]:
from scipy.special import legendre
import numpy as np
from gen.gen_gaussQuadCalc import *
from gen.gen_interpFunction import *

def detCheck(a,b):
    s0, s1 = interpLagGLQ(a,a)
    s00 = np.outer(s0,s0)
    s01 = np.outer(s0,s1)
    s11 = np.outer(s1,s1)
    dets00 = np.linalg.det(s00*b)
    dets01 = np.linalg.det(s01*b)
    dets11 = np.linalg.det(s11*b)

    if dets00 != 0.0:
        print(f'Determinant of s00 not euqal to 0 for gauss points ={a}')
        print('dets00=',dets00)
        # print('s00=',s00)
    if dets01 != 0.0:
        print(f'Determinant of s01 not euqal to 0 for gausspoints ={a}')
        print('dets01=',dets01)
    if dets11 != 0.0:
        print(f'Determinant of s11 not euqal to 0 for gauss points ={a}')
        print('dets11=',dets11)

In [47]:
from scipy.special import legendre
points = 4
polyOrder = points-1
dervPolyOrder = legendre(polyOrder).deriv()

rootsDervPolyOrder = dervPolyOrder.roots
rootsDervPolyOrder.sort()

specNodes = np.insert(rootsDervPolyOrder, [0,len(rootsDervPolyOrder)],[-1,1])
specNodes

array([-1.       , -0.4472136,  0.4472136,  1.       ])

In [22]:
detCheck(4,1)

Determinant of s00 not euqal to 0 for gauss points =4
dets00= -1.176133583717821e-262
Determinant of s01 not euqal to 0 for gausspoints =4
dets01= -7.20107556470272e-255
Determinant of s11 not euqal to 0 for gauss points =4
dets11= 1.7574529681652876e-247


In [28]:
interpLagGLQ(4,4)

(array([[ 0.62994317, -0.0706948 , -0.03482132,  0.04700152],
        [ 0.47255875,  0.97297619,  0.13253993, -0.14950343],
        [-0.14950343,  0.13253993,  0.97297619,  0.47255875],
        [ 0.04700152, -0.03482132, -0.0706948 ,  0.62994317]]),
 array([[-2.34183742, -0.51670214,  0.33325047, -0.18899664],
        [ 2.78794489, -0.48795249, -1.3379051 ,  0.63510411],
        [-0.63510411,  1.3379051 ,  0.48795249, -2.78794489],
        [ 0.18899664, -0.33325047,  0.51670214,  2.34183742]]))

In [46]:
a = interpLagGLQ(4,4)[0]
np.sum(a, axis=0)  # Each should be close to 1


array([1., 1., 1., 1.])

In [27]:
import numpy as np
from numpy.polynomial.legendre import Legendre, leggauss

def gll_nodes(degree):
    """
    Returns Gauss-Lobatto-Legendre nodes (for interpolation).
    """
    if degree < 1:
        raise ValueError("degree must be >=1")
    P = Legendre.basis(degree)
    dP = P.deriv()
    # Lobatto nodes: endpoints and roots of derivative
    interior = np.sort(dP.roots()) if degree > 1 else []
    return np.concatenate(([-1.0], interior, [1.0]))

def barycentric_weights(nodes):
    """
    Compute barycentric weights for Lagrange interpolation.
    """
    n = len(nodes)
    w = np.ones(n)
    for j in range(n):
        for k in range(n):
            if k != j:
                w[j] /= (nodes[j] - nodes[k])
    return w

def lagrange_basis_and_derivative(nodes, weights, x_eval):
    """
    Calculates Lagrange basis and its derivative at x_eval.
    Returns arrays of shape (len(x_eval), len(nodes))
    """
    n = len(nodes)
    m = len(x_eval)
    L = np.zeros((m, n))
    dL = np.zeros((m, n))
    for i, x in enumerate(x_eval):
        # Special case: if x == nodes[j]
        if np.any(np.abs(x - nodes) < 1e-14):
            j = np.argmin(np.abs(x - nodes))
            L[i, j] = 1.0
            # Derivative at the node
            for k in range(n):
                if k != j:
                    dL[i, k] = weights[k] / (weights[j] * (nodes[j] - nodes[k]))
            dL[i, j] = -np.sum(dL[i, :])
            continue
        # General case
        denom = 0.0
        numer = np.zeros(n)
        for j in range(n):
            numer[j] = weights[j] / (x - nodes[j])
            denom += numer[j]
        L[i, :] = numer / denom

        # Derivative: see Higham (2004), Berrut/Trefethen (2004)
        t = np.sum(L[i, :] / (x - nodes))
        dL[i, :] = L[i, :] * (-1/(x - nodes) + t)
    return L, dL

def example():
    # ============= User params =============
    degree = 3        # Cubic interpolation (4 nodes: [-1, x1, x2, 1])
    ngqp = 4          # Evaluate at 4 Gauss points
    # =======================================

    # Get interpolation nodes and barycentric weights
    nodes = gll_nodes(degree)
    weights = barycentric_weights(nodes)

    # Get Gauss quadrature points
    gauss_x, gauss_w = leggauss(ngqp)

    # Build basis and derivatives at Gauss points
    L, dL = lagrange_basis_and_derivative(nodes, weights, gauss_x)

    # --- Example for f(x) = x^3 ---
    # Get function values at interpolation nodes
    f_nodes = nodes**3
    # Interpolated value at Gauss points
    f_at_gauss = L @ f_nodes
    # Interpolated derivative at Gauss points
    df_at_gauss = dL @ f_nodes

    # Compare with true values
    f_true = gauss_x**3
    df_true = 3 * gauss_x**2

    print("Gauss points:",           gauss_x)
    print("Nodes:",                 nodes)
    print("Function at nodes:",     f_nodes)
    print("Interp. at Gauss:",      f_at_gauss)
    print("True at Gauss:",         f_true)
    print("Max error:",    np.max(np.abs(f_at_gauss - f_true)))
    print("Interp. derivative:",    df_at_gauss)
    print("True derivative:",       df_true)
    print("Max error d:", np.max(np.abs(df_at_gauss - df_true)))

if __name__ == "__main__":
    example()

Gauss points: [-0.86113631 -0.33998104  0.33998104  0.86113631]
Nodes: [-1.        -0.4472136  0.4472136  1.       ]
Function at nodes: [-1.         -0.08944272  0.08944272  1.        ]
Interp. at Gauss: [-0.63858058 -0.03929743  0.03929743  0.63858058]
True at Gauss: [-0.63858058 -0.03929743  0.03929743  0.63858058]
Max error: 1.1102230246251565e-16
Interp. derivative: [2.22466724 0.34676133 0.34676133 2.22466724]
True derivative: [2.22466724 0.34676133 0.34676133 2.22466724]
Max error d: 4.440892098500626e-16


In [51]:
import numpy as np
np.round(4.5).astype(int)

np.int64(4)